# Make synthetic data for a sample containing different materials

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipp as sc
import plopp as pp
from tof.utils import wavelength_to_energy, energy_to_wavelength
import tof

rng = np.random.default_rng(12345)

In [ ]:
import NCrystal as NC

NMAT = 36

xsecs = {}
for name in NC.browseFiles():
    raw = name.name
    if raw == "void.ncmat":
        print("void found: stopping")
        break
    a = raw.removesuffix(".ncmat")
    b = a.split("_")
    for s in b:
        if s.startswith("sg"):
            continue
        if all(
            x not in s
            for x in (
                "Glass",
                "Bromide",
                "Carbide",
                "Nitride",
                "Gas",
                "STP",
                "Epoxy",
                "Araldite",
                "Kapton",
                "Nylon",
                "Poly",
                "PEEK",
                "PVC",
                "Rubber",
            )
        ):
            new = s
            break
    if "HeavyWater" in new:
        new = "D2O"
    if "LiquidWater" in new:
        new = "H2O"

    # Only keep short names
    if len(new) < 3:
        xsecs[new] = NC.createScatter(raw)

    if len(xsecs) >= NMAT:
        break

In [ ]:
# shape of grid based on number of materials
nrows = int(np.ceil(np.sqrt(len(xsecs))))
ncols = nrows

materials = {}
i = j = 0
for key, xs in xsecs.items():
    materials[key] = {"loc": (i, j), "xs": xs}
    i += 1
    if i >= nrows:
        i = 0
        j += 1
materials

In [ ]:
nrows, ncols

In [ ]:
len(materials)

In [ ]:
from scipy.interpolate import interp1d

wmin = 0.05
wmax = 15.0

# 1. Pre-compute 1D transmission curves for each material
W_grid = np.linspace(wmin, wmax, 300)

background = 3.0

thickness = 1.0

# Background (material index 0)
transmission_curves = {
    0: np.exp(-np.full_like(W_grid, fill_value=background) * thickness)
}

# Each material
for i, (mat, data) in enumerate(materials.items()):
    mat_idx = i + 1
    # mu = data["material"].mu(E_grid)
    mu = np.asarray(data["xs"].xsect(wl=W_grid))
    mu = mu / mu.max()  # Normalize to max of 1 for similar intensity across materials
    mu = mu * rng.uniform(0.8, 1.2)
    transmission_curves[mat_idx] = np.exp(-mu * thickness)

# 2. Create interpolators
interpolators = {
    idx: interp1d(W_grid, trans, bounds_error=False, fill_value=0)
    for idx, trans in transmission_curves.items()
}

In [ ]:
from PIL import Image, ImageDraw, ImageFont


def make_material_patch(text: str, dx, pad, fontsize=12) -> np.ndarray:
    img = Image.new("L", (dx, dx), 0)
    font = ImageFont.truetype("DejaVuSans.ttf", fontsize)

    draw = ImageDraw.Draw(img)
    for i in range(0, dx, fontsize):
        draw.text((0, i), (text + " ") * 40, fill=255, font=font)
    # Create a circular mask
    mask = Image.new("L", (dx, dx), 0)
    mask_draw = ImageDraw.Draw(mask)
    mask_draw.ellipse((0, 0, dx - 1, dx - 1), fill=255)
    # Keep only pixels inside the circle
    img = Image.composite(img, Image.new("L", (dx, dx), 0), mask)

    a = np.array(img)
    # return a, img
    # print(a.min(), a.max())
    patch = np.where(a > 0, 1, 0).astype(int)
    out = np.zeros((dx + pad, dx + pad))
    out[pad // 2 : pad // 2 + dx, pad // 2 : pad // 2 + dx] = patch
    return np.flipud(out)

In [ ]:
dx = 256
pad = 40

xx = []
yy = []
ww = []
ee = []

# Generate events
N = 5_000_000

for i, (mat, data) in enumerate(materials.items()):
    print("Making patch for material", mat)
    row, col = data["loc"]

    patch = make_material_patch(mat, dx, pad, fontsize=10)

    x = rng.uniform(0, patch.shape[1], N)
    y = rng.uniform(0, patch.shape[0], N)

    # Uniform wavelengths for now
    wav = rng.uniform(wmin, wmax, N)

    # 3. Apply to events
    mat_indices = patch[y.astype(int), x.astype(int)]

    # Compute transmission for each event using its material's curve
    transmission = np.zeros(N)

    # 0: background
    mask = mat_indices == 0
    transmission[mask] = interpolators[0](wav[mask])

    # 1: material
    inv = ~mask
    transmission[inv] = interpolators[i + 1](wav[inv])

    keep = rng.random(N) < transmission

    xx.append(x[keep] + col * (dx + pad))
    yy.append(y[keep] + row * (dx + pad))

    wav_keep = wav[keep]
    ww.append(wav_keep)

    wavelength = sc.array(dims=["event"], values=wav_keep, unit="angstrom")
    energy = wavelength_to_energy(wavelength)

    if rng.random() < 0.1:
        scale = rng.uniform(0.001, 0.015)
        if rng.random() < 0.5:
            print("material", mat, "is gamma")
            print((0.0, scale), len(wav_keep))
            delta_e = rng.standard_gamma(rng.uniform(1.5, 4.5), len(wav_keep)) * scale
        else:
            print("material", mat, "is NORMAL")
            delta_e = rng.normal(scale=scale, size=len(wav_keep))
    else:
        delta_e = 0.0

    initial_energy = energy.values + delta_e
    ee.append(initial_energy)

In [ ]:
nevents = sum(len(a) for a in xx)
nevents

In [ ]:
image_width = (dx + pad) * ncols

In [ ]:
import numpy as np


def make_number_of_reflections(wavelengths, x, y, seed=None):
    rng = np.random.default_rng(seed)

    wavelengths = np.asarray(wavelengths)
    x = np.asarray(x)
    y = np.asarray(y)

    # Base relationship: longer wavelength -> more reflections
    mean = 1.5 + 2.0 * wavelengths**1.5

    # ------------------------------------------------------------
    # Position-dependent effect
    # ------------------------------------------------------------

    # Normalize x/y to the detector half-width.
    # Assuming a 10 cm x 10 cm detector, with x/y in metres.
    # half_width = 0.05
    half_width = 1.0

    r = np.sqrt((x / half_width) ** 2 + (y / half_width) ** 2)

    # Make the effect gentle:
    #
    # centre  -> multiplier ~1.0
    # edge    -> multiplier ~1.2
    #
    # Clip because corners can have r > 1.
    edge_effect = 1.0 + 0.2 * np.clip(r, 0, 1) ** 2

    mean *= edge_effect

    # ------------------------------------------------------------
    # Asymmetric noise
    # ------------------------------------------------------------

    noise = rng.gamma(shape=5, scale=1 / 5, size=len(wavelengths))

    reflections = mean * noise * 0.1

    reflections = np.rint(reflections).astype(int)

    return np.maximum(reflections, 0)

In [ ]:
def make_birth_coordinates(wavelengths, seed=None):
    """
    Generate synthetic neutron birth positions on a simplified
    butterfly-like moderator.

    Cold region:
        Large, broad, gently curved surface on the left.

    Thermal region:
        Smaller, strongly curved surface on the right.

    For every (x, y), the neutron is placed on whichever surface
    has the largest z, preventing birth positions from occurring
    inside the moderator.

    Coordinates:
        x = horizontal
        y = vertical
        z = direction toward the guide

    Returns
    -------
    x, y, z : ndarray
        Birth coordinates in metres.

    is_thermal : ndarray of bool
        True for thermal, False for cold.
    """

    rng = np.random.default_rng(seed)

    wavelengths = np.asarray(wavelengths)
    n = len(wavelengths)

    # ============================================================
    # Moderator dimensions
    # ============================================================

    cold_x = -0.045
    cold_width = 0.14
    cold_height = 0.065

    thermal_x = +0.040
    thermal_width = 0.065
    thermal_height = 0.050

    # ============================================================
    # Wavelength-dependent thermal probability
    # ============================================================

    wavelength_thermal = 1 / (1 + np.exp((wavelengths - 3.5) / 0.8))

    # ============================================================
    # Sample x/y positions
    # ============================================================

    initial_thermal = rng.random(n) < wavelength_thermal

    x = np.empty(n)
    y = np.empty(n)

    # ------------------------------------------------------------
    # Cold spatial distribution
    # ------------------------------------------------------------

    cold_mask = ~initial_thermal
    nc = np.sum(cold_mask)

    x[cold_mask] = rng.normal(cold_x, cold_width / 3, nc)

    y[cold_mask] = rng.normal(0, cold_height / 3, nc)

    # ------------------------------------------------------------
    # Thermal spatial distribution
    # ------------------------------------------------------------

    thermal_mask = initial_thermal
    nt = np.sum(thermal_mask)

    x[thermal_mask] = rng.normal(thermal_x, thermal_width / 3, nt)

    y[thermal_mask] = rng.normal(0, thermal_height / 3, nt)

    # ============================================================
    # Spatial thermal preference
    # ============================================================

    spatial_thermal = 1 / (1 + np.exp(-x / 0.012))

    thermal_probability = 0.55 * wavelength_thermal + 0.45 * spatial_thermal

    is_thermal = rng.random(n) < thermal_probability

    # ============================================================
    # Calculate BOTH surfaces
    # ============================================================

    # ------------------------------------------------------------
    # Cold surface
    # ------------------------------------------------------------

    dx_cold = x - cold_x
    dy_cold = y

    z_cold = 0.025 - 2.8 * dx_cold**2 - 5.0 * dy_cold**2

    z_cold += 0.0025 * np.sin(2 * np.pi * dx_cold / cold_width) + 0.0015 * np.sin(
        2 * np.pi * dy_cold / cold_height
    )

    # ------------------------------------------------------------
    # Thermal surface
    # ------------------------------------------------------------

    dx_thermal = x - thermal_x
    dy_thermal = y

    z_thermal = 0.030 - 8.0 * dx_thermal**2 - 14.0 * dy_thermal**2

    z_thermal += 0.0015 * np.sin(
        2 * np.pi * dx_thermal / thermal_width
    ) + 0.0010 * np.sin(2 * np.pi * dy_thermal / thermal_height)

    # ============================================================
    # Choose the outermost surface
    # ============================================================

    # Whichever surface has the larger z is the surface visible
    # from the guide.

    use_thermal_surface = z_thermal > z_cold

    z = np.where(use_thermal_surface, z_thermal, z_cold)

    # ============================================================
    # Small surface roughness
    # ============================================================

    z += rng.normal(0, 0.0003, n)

    # ============================================================
    # Make source identity consistent with visible surface
    # ============================================================

    # A neutron on the thermal surface should be thermal, and
    # vice versa.

    return x, y, z

In [ ]:
import scipp as sc
from tof.utils import wavelength_to_energy, energy_to_wavelength, wavelength_to_speed

dist_from_source = sc.scalar(60.5, unit="m")

events = sc.DataArray(
    data=sc.ones(sizes={"event": nevents}),
    coords={
        "x": sc.array(
            dims=["event"],
            values=(np.concatenate(xx) - 0.5 * image_width) * 1.0e-3,
            unit="m",
        ),
        "y": sc.array(
            dims=["event"],
            values=(np.concatenate(yy) - 0.5 * image_width) * 1.0e-3,
            unit="m",
        ),
        "z": sc.array(
            dims=["event"],
            values=np.full(nevents, fill_value=dist_from_source.value),
            unit=dist_from_source.unit,
        ),
        "wavelength": sc.array(
            dims=["event"], values=np.concatenate(ww), unit="angstrom"
        ),
        "initial_energy": sc.array(
            dims=["event"], values=np.concatenate(ee), unit="meV"
        ),
    },
)

events.coords["initial_wavelength"] = energy_to_wavelength(
    events.coords["initial_energy"]
)
events.coords["final_energy"] = wavelength_to_energy(events.coords["wavelength"])
events.coords["energy_transfer"] = (
    events.coords["initial_energy"] - events.coords["final_energy"]
)


refl = make_number_of_reflections(
    events.coords["wavelength"].values,
    x=events.coords["x"].values,
    y=events.coords["y"].values,
)
events.coords["reflections"] = sc.array(dims=["event"], values=refl)

xsource, ysource, zsource = make_birth_coordinates(events.coords["wavelength"].values)
events.coords["birth_x"] = sc.array(dims=["event"], values=xsource, unit="m")
events.coords["birth_y"] = sc.array(dims=["event"], values=ysource, unit="m")
events.coords["birth_z"] = sc.array(dims=["event"], values=zsource, unit="m")

s = tof.Source(facility="ess", neutrons=nevents).data.squeeze()
events.coords["birth_time"] = s.coords["birth_time"]
events.coords["toa"] = (
    dist_from_source / wavelength_to_speed(events.coords["wavelength"])
).to(unit=events.coords["birth_time"].unit) + events.coords["birth_time"]

events

In [ ]:
events.hist(y=256, x=256).plot(aspect="equal")

In [ ]:
events.hist(toa=300).plot()

In [ ]:
events.hist(reflections=300, wavelength=300).plot(logc=True)

In [ ]:
binned = events.bin(y=6, x=6)
binned

In [ ]:
a = binned["x", 0]["y", 0]
b = binned["x", 3]["y", 2]

In [ ]:
pp.plot(
    {
        "a": a.hist(wavelength=300),
        "b": b.hist(wavelength=300),
    }
)

In [ ]:
events.save_hdf5("special_imaging_sample.h5")